# Helmholtz eigenvalue problem in a rectangle

$$
\mathbb{S}
\begin{cases}
\Omega = [0, L_x] \times [0, L_y] \\
u_{\text{D}}\vert_{\partial\Omega}=0 \\
f = 0 \\
\end{cases}
$$

In [1]:
import numpy as np
from dolfinx.fem import FunctionSpace

from lucifex.mesh import rectangle_mesh, mesh_boundary
from lucifex.solver import (
    evp, BoundaryConditions, OptionsSLEPc,
)
from lucifex.viz import plot_colormap, save_figure
from lucifex.pde.eigen import helmholtz


Lx = 1.0
Ly = 1.0
Nx = 32
Ny = 32

mesh = rectangle_mesh(Lx, Ly, Nx, Ny)
boundary = mesh_boundary(
        mesh, 
        {
            "left": lambda x: x[0],
            "right": lambda x: x[0] - Lx,
            "lower": lambda x: x[1],
            "upper": lambda x: x[1] - Ly,
        },
)
bcs = BoundaryConditions(('dirichlet', boundary.union, 0.0))
fs = FunctionSpace(mesh, ('P', 1))

nev = 10
slepc = OptionsSLEPc(
    eps_tol=1e-10,
    eps_target=0.0,
    eps_nev=nev,
    eps_ncv=50,
    eps_max_it=1000,
    eps_which='smallest_real',
)
u_solver = evp(helmholtz, bcs, slepc)(fs)
u_solver.solve()

AttributeError: 'FunctionSpace' object has no attribute 'function_space'

In [2]:
sqrt = lambda x: np.sqrt(np.real(x)) if np.real(x) > 0 else None
k_values = [sqrt(-e) for e in u_solver.eigenvalues]
k_values

NameError: name 'u_solver' is not defined

In [3]:
for k, u in zip(k_values, u_solver.eigenfunctions):
    if k is not None:
        fig, ax = plot_colormap(u, title=f'$k={k}$')
        save_figure(f'k={k:.3f}\nu', thumbnail=(k is k_values[-1]))(fig)

NameError: name 'k_values' is not defined